#Get data from Kaggle

In [2]:
!curl -L -o dataset.zip https://www.kaggle.com/api/v1/datasets/download/birdy654/cifake-real-and-ai-generated-synthetic-images

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  104M  100  104M    0     0  11.3M      0  0:00:09  0:00:09 --:--:-- 16.0M


#Unzip Data

In [3]:
import zipfile

file = "dataset.zip"

with zipfile.ZipFile(file, 'r') as zf:
  zf.extractall()

#Create Validation Folder

In [4]:
try:
  !mkdir validation
  %cd validation
  !mkdir REAL
  !mkdir FAKE
except FileExistsError:
  print("File already exists")

/content/validation


#Move 20k files for validation

In [5]:
import shutil
from pathlib import Path

train_real = Path("/content/train/REAL")
train_fake = Path("/content/train/FAKE")
val_real = Path("/content/validation/REAL")
val_fake = Path("/content/validation/FAKE")

counter = 0
for child in train_real.iterdir():
  shutil.move(child, val_real)
  counter += 1
  if counter == 10000:
      break

counter = 0
for child in train_fake.iterdir():
  shutil.move(child, val_fake)
  counter += 1
  if counter == 10000:
      break

#Build CNN

In [6]:
from tensorflow import keras
from keras import layers

input = keras.Input(shape=(32,32,3))
x = layers.Conv2D(filters=32, kernel_size=3, activation='relu')(input)
x = layers.MaxPooling2D()(x)
x = layers.Conv2D(filters=32, kernel_size=3, activation='relu')(x)
x = layers.MaxPooling2D()(x)
x = layers.Flatten()(x)
x = layers.Dense(64, activation='relu')(x)
output = layers.Dense(1, activation='sigmoid')(x)

model = keras.Model(inputs=input, outputs=output)

model.summary()



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 30, 30, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 15, 15, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 13, 13, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 6, 6, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1152)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        73,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 84,001 (328.13 KB)

 Trainable params: 84,001 (328.13 KB)

 Non-trainable params: 0 (0.00 B)

#Train Model

In [15]:
training = keras.preprocessing.image_dataset_from_directory("/content/train", image_size=(32, 32), batch_size=128)
test = keras.preprocessing.image_dataset_from_directory("/content/test", image_size=(32, 32), batch_size=128)
validation = keras.preprocessing.image_dataset_from_directory("/content/validation", image_size=(32, 32), batch_size=128)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="real_or_fake_best.keras",
        save_best_only=True,
        monitor="val_loss"
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        min_delta=0.001
    ),
]

model.compile(loss="binary_crossentropy",
              optimizer="adam",
              metrics=["accuracy"],
              )

history = model.fit(
    training,
    epochs=30,
    validation_data=validation,
    callbacks=callbacks
)

test_model = keras.models.load_model('real_or_fake_best.keras')
loss, accuracy = test_model.evaluate(test)
print(f"Loss: {loss}, Accuracy: {accuracy}")

Found 80000 files belonging to 2 classes.
Found 20000 files belonging to 2 classes.
Found 20000 files belonging to 2 classes.
Epoch 1/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - accuracy: 0.9340 - loss: 0.1719 - val_accuracy: 0.8906 - val_loss: 0.3002
Epoch 2/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.9351 - loss: 0.1699 - val_accuracy: 0.9199 - val_loss: 0.2424
Epoch 3/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 19s 15ms/step - accuracy: 0.9398 - loss: 0.1573 - val_accuracy: 0.9203 - val_loss: 0.2744
Epoch 4/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.9409 - loss: 0.1561 - val_accuracy: 0.9198 - val_loss: 0.2748
Epoch 5/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 12s 19ms/step - accuracy: 0.9412 - loss: 0.1539 - val_accuracy: 0.9225 - val_loss: 0.2417
Epoch 6/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.9373 - loss: 0.1620 - val_accuracy: 0.9169 - val_loss: 0.2761
Epoch 7/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 17ms/step - accuracy: 0.9428 - loss: 0.1509 - val_